<a href="https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [35]:
import pandas as pd

url = "https://raw.githubusercontent.com/estuchis21/ml-internship-week1/refs/heads/main/data/raw/content_refresh_anonymized.csv"

original_df = pd.read_csv(url)
original_df["days_with_impressions"].sort_values()

,days_with_impressions
7342,1
29992,1
16106,1
22986,1
19920,1
...,...
4,88
5,88
17,88
16,88


## 2. Target or proxy

The target is a proxy label created by filtering data (shows only pages worth refreshing).

A positive example (target=1) represents a content page with declining performance but enough visibility to justify editorial review.

The label is not a real outcome of a refresh experiment. It is a decision-support proxy created from historical observations.

In [64]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
original_df["target"] = (
    (original_df["trend_direction"] == "down")
    &
    (original_df["trend_pct"] <= -20)
    &
    (original_df["impressions_90d"] >= 500)
).astype(int)

original_df["target"].value_counts()

X = original_df.drop(
    columns=[
        "trend_pct",
        "trend_direction",
        "target"
    ]
)


In [67]:
original_df[["content_id","target"]].sort_values(by="target", ascending=False).head(100)

,content_id,target
6832,content_cb44903ae1d9,1
6833,content_89c19a592e27,1
6834,content_ed30e6d83dc7,1
6835,content_8f0f2e1a797b,1
11069,content_f59e2b217781,1
...,...,...
20010,content_b2b2f104d32c,1
22586,content_eb178839a463,1
22585,content_00cd5e91c97a,1
14941,content_801e26d91ba6,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success metric

I will use Recall as the primary success metric.

With this measure, I am trying to find how many true positives are detected by the model. Hence, false negatives are costly because missing a high-value declining page means losing the opportunity to improve its performance.

A good model would achieve a recall above 0.80, meaning it detects at least 80% of the pages that should be prioritized for review.

In [66]:
# Check target distribution before training

target_distribution = original_df["target"].value_counts(normalize=True)

print(target_distribution)

print("\nNumber of positive cases:")
print(original_df["target"].sum())

target
0    0.667967
1    0.332033
Name: proportion, dtype: float64

Number of positive cases:
9961


The target distribution shows that approximately one third of pages are labeled as refresh opportunities. This provides enough positive examples for a classification task while keeping the problem realistic.

## 4. The unit of analysis, as a real dataframe

1 fila = 1 content page

In [69]:
original_df[
    [
        "content_id",
        "impressions_90d",
        "trend_pct",
        "avg_position",
        "days_since_last_update",
        "target"
    ]
].head()

,content_id,impressions_90d,trend_pct,avg_position,days_since_last_update,target
0,content_304f48230142,3803,-41.4,10.6,20,1
1,content_a1fb4e703a9e,15320,-57.7,20.3,25,1
2,content_9aa793d4d895,12581,-60.9,36.5,20,1
3,content_331d6c4de07b,11751,-13.8,6.2,22,0
4,content_d99b7a2d90ca,19140,-34.7,44.0,14,1


## Why ML beats a fixed rule here

A fixed rule would only consider a small number of conditions, such as traffic decline or a specific trend threshold.

However, content performance depends on multiple factors which interact with each other. A page with high impressions, a recent decline, good ranking position and low engagement may represent a different opportunity than a page with low visibility.

Machine learning can combine multiple features and learn patterns from historical observations instead of relying on manually chosen thresholds.

The model does not replace editorial decisions; it helps create a more informed priority list for human review.

In [71]:
features = [
    "impressions_90d",
    "trend_pct",
    "avg_position",
    "ctr",
    "engagement_rate",
    "days_since_last_update"
]

original_df[features].describe()

,impressions_90d,trend_pct,avg_position,ctr,engagement_rate,days_since_last_update
count,30000.000000,26612.000000,30000.00000,30000.000000,30000.000000,30000.000000
mean,5200.366300,-4.785969,16.34238,0.510733,2.534520,46.098300
std,16838.019547,473.861780,15.21679,3.279162,8.310096,42.078709
min,1.000000,-100.000000,0.00000,0.000000,0.000000,1.000000
25%,81.000000,-62.600000,6.20000,0.000000,0.000000,20.000000
50%,731.000000,-33.500000,10.80000,0.070000,0.000000,20.000000
75%,3615.250000,0.000000,22.30000,0.290000,1.350000,104.000000
max,517715.000000,44900.000000,245.00000,100.000000,100.000000,373.000000
